# Vanilla Explore

Figure out a stable version of vanilla model as a baseline.

In [ ]:
import json

from torch import nn
import torch

from blockbuster.models.config import TrainConfig
from blockbuster.models.vanillas import (
    BaselineGPT,
    TransformerBlock,
    CausalSelfAttention,
)
from blockbuster.data import build_dataset, build_test_dataset

/Users/drwho/ghb/blockbuster/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
cfg = TrainConfig()
for key, value in cfg.model_dump().items():
    print(f"{key}: {value}")


block_size: 512
max_train_rows: 100000
test_rows: 500
dataset: HuggingFaceFW/fineweb
batch_size: 8
lr: 0.0003
num_epochs: 1
log_every: 50
n_layer: 12
n_head: 6
hidden_size: 768
vocab_size: 50257


## Model

In [3]:
model = BaselineGPT(
    vocab_size=cfg.vocab_size,
    n_positions=cfg.block_size,
    hidden_size=cfg.hidden_size,
    n_layer=cfg.n_layer,
    n_head=cfg.n_head,
)

In [4]:
def get_param_count(module: nn.Module) -> int:
    return sum(p.nelement() for p in module.parameters())

def large_int(x: int) -> str:
    return f"{x:,}"

In [5]:
model

BaselineGPT(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(512, 768)
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (c_attn): Linear(in_features=768, out_features=2304, bias=True)
        (c_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (c_fc): Linear(in_features=768, out_features=3072, bias=True)
        (c_proj): Linear(in_features=3072, out_features=768, bias=True)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [6]:
total_parameter: int = get_param_count(model)
print(f"total: {large_int(total_parameter)}")

wpe_param_ct: int = get_param_count(model.wpe)
wte_param_ct: int = get_param_count(model.wte)
print(f"embedding: wpe:{large_int(wpe_param_ct)}, wte:{large_int(wte_param_ct)}, total:{large_int(wpe_param_ct + wte_param_ct)}, {(wpe_param_ct + wte_param_ct) / total_parameter * 100:.2f}%")

lm_head_param_ct: int = get_param_count(model.lm_head)
print(f"lm_head: {large_int(lm_head_param_ct)}, {lm_head_param_ct / total_parameter * 100:.2f}%")

transformer_blocks_param_ct: int = get_param_count(model.blocks)
print(f"transformer_blocks: {large_int(transformer_blocks_param_ct)}, {transformer_blocks_param_ct / total_parameter * 100:.2f}%")

for name, module in model.named_modules():
    if isinstance(module, TransformerBlock):
        transformer_param_ct: int = get_param_count(module)
        print(f"{name}: {large_int(transformer_param_ct)}, {transformer_param_ct / total_parameter * 100:.2f}%")
    if isinstance(module, CausalSelfAttention):
        attention_param_ct: int = get_param_count(module)
        print(f"{name}: {large_int(attention_param_ct)}, {attention_param_ct / total_parameter * 100:.2f}%")


total: 162,643,968
embedding: wpe:393,216, wte:38,597,376, total:38,990,592, 23.97%
lm_head: 38,597,376, 23.73%
transformer_blocks: 85,054,464, 52.29%
blocks.0: 7,087,872, 4.36%
blocks.0.attn: 2,362,368, 1.45%
blocks.1: 7,087,872, 4.36%
blocks.1.attn: 2,362,368, 1.45%
blocks.2: 7,087,872, 4.36%
blocks.2.attn: 2,362,368, 1.45%
blocks.3: 7,087,872, 4.36%
blocks.3.attn: 2,362,368, 1.45%
blocks.4: 7,087,872, 4.36%
blocks.4.attn: 2,362,368, 1.45%
blocks.5: 7,087,872, 4.36%
blocks.5.attn: 2,362,368, 1.45%
blocks.6: 7,087,872, 4.36%
blocks.6.attn: 2,362,368, 1.45%
blocks.7: 7,087,872, 4.36%
blocks.7.attn: 2,362,368, 1.45%
blocks.8: 7,087,872, 4.36%
blocks.8.attn: 2,362,368, 1.45%
blocks.9: 7,087,872, 4.36%
blocks.9.attn: 2,362,368, 1.45%
blocks.10: 7,087,872, 4.36%
blocks.10.attn: 2,362,368, 1.45%
blocks.11: 7,087,872, 4.36%
blocks.11.attn: 2,362,368, 1.45%


## Hooks

In [7]:
from time import time
import torch

In [30]:
from dataclasses import dataclass, field
@dataclass
class TensorStats:
    numel: int = 0
    mean: float = 0.0
    max: float = 0.0
    min: float = 0.0
    std: float = 0.0
    
    def dict(self):
        return {
            "numel": self.numel,
            "mean": self.mean,
            "max": self.max,
            "min": self.min,
            "std": self.std,
        }


In [31]:
def extract_activation(activation, prefix: str = "") -> dict[str, dict]:
    results: dict[str, TensorStats] = {}
    if isinstance(activation, torch.Tensor):
        t = activation.float()
        results[prefix] = TensorStats(
            numel=activation.numel(),
            mean=t.mean().item(),
            max=t.max().item(),
            min=t.min().item(),
            std=t.std().item() if t.numel() > 1 else 0.0,
        ).dict()
        return results
    if hasattr(activation, "items"):
        for key, value in activation.items():
            child_prefix = f"{prefix}.{key}" if prefix else key
            results.update(extract_activation(value, child_prefix))
    if isinstance(activation, (list, tuple)):
        for i, item in enumerate(activation):
            child_prefix = f"{prefix}.{i}" if prefix else str(i)
            results.update(extract_activation(item, child_prefix))
    return results

In [32]:
extract_activation({
    "a":torch.randn(10, 10),
    "b":[torch.randn(10, 20), torch.randn(10, 30)],
    })

{'a': {'numel': 100,
  'mean': 0.19880172610282898,
  'max': 3.581225633621216,
  'min': -2.0708508491516113,
  'std': 0.9912310838699341},
 'b.0': {'numel': 200,
  'mean': 0.03132602944970131,
  'max': 2.676464080810547,
  'min': -3.0767323970794678,
  'std': 1.033299207687378},
 'b.1': {'numel': 300,
  'mean': 0.0062117306515574455,
  'max': 2.5474040508270264,
  'min': -3.0055932998657227,
  'std': 0.9362375736236572}}

In [33]:
class ModelHookOneOff:
    """
    Any hooks we place, will run only once, afterward the hook will be removed.
    """
    def __init__(
        self,
        model: nn.Module,
        target_modules:list| None = None,
    ):
        self.model = model
        self.target_modules = target_modules
        self.metrics = {}

    def _register_fwd_hook(self, name, module: nn.Module):
        module_name = module.__class__.__name__
        forward_clean_up_list: list = []
        m = f"{module_name}|{name}|fwd"
        self.metrics[m] = dict()
        
        def forward_pre_hook(module, inputs):
            with torch.no_grad():
                self.metrics[m]["fwd_input_stats"] = extract_activation(inputs)
                self.metrics[m]["fwd_start_ts"] = time()
            
        def forward_after_hook(module, inputs, output):
            with torch.no_grad():
                self.metrics[m]["fwd_end_ts"] = time()
                self.metrics[m]["fwd_duration"] = self.metrics[m]["fwd_end_ts"] - self.metrics[m]["fwd_start_ts"]
                self.metrics[m]["fwd_output_stats"] = extract_activation(output)

            # REMOVE forward hooks ❌🪝
            for hook in forward_clean_up_list:
                hook.remove()

        # Arm forward hooks 🔫🪝
        forward_clean_up_list.append(module.register_forward_pre_hook(forward_pre_hook))
        forward_clean_up_list.append(module.register_forward_hook(forward_after_hook))
    
    def wire_forward(self,):
        """
        Wire forward hooks to the model.
        """

        for name, module in self.model.named_modules():
            module_name = module.__class__.__name__
            if self.target_modules is not None and not isinstance(module, tuple(self.target_modules)):
                continue

            self._register_fwd_hook(name, module)


    def _register_bwd_hook(self, name, module: nn.Module):
        module_name = module.__class__.__name__
        backward_clean_up_list: list = []

        m = f"{module_name}|{name}|bwd"
        self.metrics[m] = dict()
        
        def backward_pre_hook(module, grad_output):
            with torch.no_grad():
                self.metrics[m]["bwd_grad_output_pre_stats"] = extract_activation(grad_output)
                self.metrics[m]["bwd_start_ts"] = time()
            
        def backward_after_hook(module, grad_input, grad_output):
            with torch.no_grad():
                self.metrics[m]["bwd_end_ts"] = time()
                self.metrics[m]["bwd_duration"] = self.metrics[m]["bwd_end_ts"] - self.metrics[m]["bwd_start_ts"]
                self.metrics[m]["bwd_grad_output_after_stats"] = extract_activation(grad_output)
                self.metrics[m]["bwd_grad_input_after_stats"] = extract_activation(grad_input)

            # REMOVE backward hooks ❌🪝
            for hook in backward_clean_up_list:
                hook.remove()

        # Arm backward hooks 🔫🪝
        backward_clean_up_list.append(module.register_backward_hook(backward_pre_hook))
        backward_clean_up_list.append(module.register_full_backward_hook(backward_after_hook))


    def wire_backward(self,):
        """
        Wire backward hooks to the model.
        """

        for name, module in self.model.named_modules():
            module_name = module.__class__.__name__
            if self.target_modules is not None and module_name not in self.target_modules:
                continue

            self._register_bwd_hook(name, module)


In [34]:
hook_one_off = ModelHookOneOff(
    model, target_modules=[
        BaselineGPT, TransformerBlock, CausalSelfAttention,
        ]
)

In [35]:
x = torch.randint(0, cfg.vocab_size, (2, 12))

hook_one_off.wire_forward()

y_ = model(x)

hook_one_off.wire_backward()

y_.logits.mean().backward()



In [ ]:
for name, metrics in list(hook_one_off.metrics.items())[:5]:
    print(name)
    print(json.dumps(metrics, indent=4))
    print()


BaselineGPT||fwd
{
    "fwd_input_stats": {
        "0": {
            "numel": 24,
            "mean": 19857.958984375,
            "max": 49615.0,
            "min": 188.0,
            "std": 14323.5224609375
        }
    },
    "fwd_start_ts": 1776149520.366976,
    "fwd_end_ts": 1776149520.770447,
    "fwd_duration": 0.4034709930419922,
    "fwd_output_stats": {}
}

TransformerBlock|blocks.0|fwd
{
    "fwd_input_stats": {
        "0": {
            "numel": 18432,
            "mean": 0.00027504118042998016,
            "max": 0.09787056595087051,
            "min": -0.10942672938108444,
            "std": 0.028174567967653275
        }
    },
    "fwd_start_ts": 1776149520.37808,
    "fwd_end_ts": 1776149520.431272,
    "fwd_duration": 0.053192138671875,
    "fwd_output_stats": {
        "": {
            "numel": 18432,
            "mean": -0.006737133022397757,
            "max": 1.6928677558898926,
            "min": -1.7358657121658325,
            "std": 0.4084097146987915
  